# 18 · 2단계 병변 보존 crop 파일럿 — 정상을 빼고, 병변 6종에서 `fixed` vs `safe`

기존 `safe-crop-pilot` Dataset 연결 → **GPU T4** 선택, Internet ON → Run All.
데이터 재업로드 불필요. **HOURS 는 남은 GPU 할당량에 맞춰** 줄이세요
(지금 값 0.9 는 이번 주 남은 1시간 7분 기준. 다 못 돌면 epoch 경계에서 이어받습니다).

## 왜 2단계인가

1단계에서 `safe`(병변 보존 random ROI)는 **두 번 졌습니다** (STEP 45 · 48). 남은 가설은
*"정상 사진엔 보존할 병변이 없어 두 클래스의 크롭 분포가 갈린다"* 였습니다.
**2단계에는 그 비대칭이 없습니다** — 넘어온 사진은 전부 병변이 있습니다.
그리고 2단계의 알려진 약점이 **위치 교란 하락 32.2%** (STEP 16 holdout) 입니다.
그래서 같은 패키지에서 **A7 을 빼고** 병변 6종만으로 같은 비교를 합니다.
사전등록: [`docs/results/STEP49_2단계_병변보존_crop_사전등록.md`](../docs/results/STEP49_2단계_병변보존_crop_사전등록.md)

## 두 팔 — 다른 건 crop 하나뿐

- **`fixed`**(대조군): 병변 bbox 를 5% 여유로 감싼 정사각 ROI(최소 320px), 가운데
- **`safe`**: 같은 ROI 를 ×1.0~1.25 키우고 **병변을 다 담는 범위 안에서** 위치를 무작위로
- 검증은 **둘 다 고정 ROI**. 같은 EfficientNetV2-S 초기값 · 384px · 6클래스 head ·
  train 10,177 / val 2,040 · 최대 5 epoch · class-weighted CE

⚠️ 배포 백본(convnextv2_base)이 아닙니다 — *"확대 실험을 할 가치가 있나"* 까지만 답합니다.

## 판정에 쓸 것 — 돌리기 전에 정했습니다

매 epoch **검증 2,040장 전부**를 두 번 봅니다: **clean**(고정 ROI) 과
**shift**(같은 ROI 를 변 길이의 20% 만큼 오른쪽·아래로, STEP 47 과 같은 정의).

    주 지표: shift 하락률 = (clean − shift) / clean macro-F1
    하한선: fixed 팔 자신의 하락률 (같은 학습·같은 val)
    후보:   safe 의 하락률이 fixed 보다 5%p 이상 작다  (교란 잡음 ±5%p)
    관문:   clean macro-F1 −0.02 이내 · 계열 4군 정확도 −0.01 이내 · A6 recall −0.03 이내

⚠️ **holdout 은 안 엽니다.** 같은 epoch 끼리만 비교합니다. 단일 seed 예비 실험입니다.
⚠️ 커버리지는 1단계 확률이 있어야 재는 값이라 **여기서는 못 잽니다** — 계열 4군 정확도로 대신합니다.

결과: `/kaggle/working/stage2_crop_pilot_resume.zip` (1단계 실험 출력과 별도).
재개: 그 ZIP 을 **비공개** Dataset 으로 연결하고 같은 노트북을 실행하세요.
`EPOCHS`·`BATCH_SIZE` 는 유지합니다.

In [ ]:
from pathlib import Path
import json, subprocess, sys, os, zipfile
HOURS = 0.9  # 이번 세션 예산. 실제 남은 GPU 시간(1h 07m)보다 여유 있게 작게 설정
EPOCHS = 5  # 재개할 때 변경하지 않기
BATCH_SIZE = 16
WORKERS = 2
CODE = Path('/kaggle/working/stage2_code')
OUT = Path('/kaggle/working/stage2_crop_pilot')
FILES = {"src/__init__.py": "\"\"\"반려견 피부질환 스크리닝 보조 모델 — 소스 패키지.\n\n⚠️ 이 프로젝트의 산출물은 수의학적 진단이 아닙니다.\n   보호자에게 \"의심 소견이 있으니 병원에 가보세요\" 수준의 안내만 제공합니다.\n   docs/cautions/03_의료AI_안전설계_원칙.md 를 반드시 읽어주세요.\n\"\"\"\n\n__version__ = \"0.1.0\"\n", "src/config.py": "\"\"\"프로젝트 전역 설정 — 모든 \"숫자\"는 여기 한 곳에 모읍니다.\n\n노트북에서 하이퍼파라미터를 직접 고치지 마세요. 여기서 고치고 노트북은 읽기만 하면\n어떤 설정으로 어떤 결과가 나왔는지 나중에 추적할 수 있습니다.\n\n    from src.config import CFG, CLASSES\n    cfg = CFG()                          # 기본값\n    cfg = CFG(img_size=384, epochs=20)   # 일부만 바꾸기\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field, asdict\nfrom pathlib import Path\nfrom typing import Any\n\n# ──────────────────────────────────────────────────────────────\n# 클래스 정의\n#\n# ⚠️ 매우 중요: 이 라벨들은 \"병명\"이 아니라 \"병변의 형태(morphology)\"입니다.\n#   A2 가 나왔다고 \"이 강아지는 지루성 피부염\" 이 아니라\n#   \"비듬·각질 형태의 병변이 보인다\" 까지가 모델이 말할 수 있는 전부입니다.\n#   같은 병변 형태가 여러 질환에서 나오고, 같은 질환이 여러 형태로 나타납니다.\n#   → docs/data/병변_6종_임상_해설.md 참고\n# ──────────────────────────────────────────────────────────────\nCLASSES: list[str] = [\"A1\", \"A2\", \"A3\", \"A4\", \"A5\", \"A6\"]\n\nCLASS_KO: dict[str, str] = {\n    \"A1\": \"구진·플라크\",\n    \"A2\": \"비듬·각질·상피성잔고리\",\n    \"A3\": \"태선화·과다색소침착\",\n    \"A4\": \"농포·여드름\",\n    \"A5\": \"미란·궤양\",\n    \"A6\": \"결절·종괴\",\n}\n\nCLASS_EN: dict[str, str] = {\n    \"A1\": \"Papule / Plaque\",\n    \"A2\": \"Scale / Crust / Epidermal collarette\",\n    \"A3\": \"Lichenification / Hyperpigmentation\",\n    \"A4\": \"Pustule / Acne\",\n    \"A5\": \"Erosion / Ulcer\",\n    \"A6\": \"Nodule / Mass\",\n}\n\n# 1단계(정상/이상) 이진 분류용 라벨.\n# ✅ 실물 확인 결과 무증상 데이터가 존재합니다 — metaData.lesions == \"A7\".\n#    유증상 26,191 / 무증상 28,042 로 거의 반반이라 2단계 모델이 가능합니다.\n#    ⚠️ 무증상 이미지가 'A1_구진_플라크' 같은 폴더 안에 들어 있습니다.\n#       폴더명이 아니라 metaData.lesions 를 봐야 합니다.\nNORMAL_LABEL = \"A7\"\nCLASS_KO[NORMAL_LABEL] = \"무증상(정상)\"\nCLASS_EN[NORMAL_LABEL] = \"Normal / Asymptomatic\"\n\n# 1단계에서 쓰는 클래스 (정상 vs 이상)\nCLASSES_STAGE1 = [NORMAL_LABEL, \"ABNORMAL\"]\n# 2단계에서 쓰는 클래스 (병변 6종) = CLASSES\n\n# 병변별 임상적 긴급도 힌트 — \"의심된다\"의 톤을 조절할 때 씁니다.\n# 진단이 아니라 안내 문구의 강도를 정하는 용도일 뿐입니다.\nURGENCY_HINT: dict[str, str] = {\n    \"A1\": \"관찰\",\n    \"A2\": \"관찰\",\n    \"A3\": \"만성 경과 가능 — 진료 권장\",\n    \"A4\": \"감염 동반 가능 — 진료 권장\",\n    \"A5\": \"피부 장벽 손상 — 조기 진료 권장\",\n    \"A6\": \"종양 감별 필요 — 조기 진료 권장\",\n}\n\n#: 긴급도 등급 (낮을수록 급하지 않음). `URGENCY_HINT` 의 문구를 순서로 옮긴 것.\n#: ⚠️ 이 순서는 `docs/data/병변_6종_임상_해설.md` 요약표의 \"긴급도\" 열이고,\n#:    **이 프로젝트의 어떤 실험보다 먼저** 적혀 있었습니다. 혼동행렬을 보고\n#:    만든 묶음이 아닙니다 — 데이터를 보고 묶으면 뭐든 좋아 보입니다.\nURGENCY_TIER: dict[str, int] = {\n    \"A1\": 0, \"A2\": 0,          # 관찰\n    \"A3\": 1, \"A4\": 1,          # 진료 권장\n    \"A5\": 2, \"A6\": 2,          # 조기 진료 권장\n}\nURGENCY_TIER_NAME: dict[int, str] = {\n    0: \"관찰\", 1: \"진료 권장\", 2: \"조기 진료 권장\",\n}\n\n#: 형태 계열 **3군** — 임상 해설의 **\"🟢 비교적 안전한 혼동\"** 목록\n#: (A1↔A4 · A5↔A6) 과 형태 설명을 그대로 따릅니다.\n#: ⚠️ **화면에 나가는 것은 이게 아니라 `MORPH_GROUP_KEEP_A6`(4군)** 입니다.\n#:    이 3군은 **실험 비교용**이라 과거 결과표와 대조가 되게 **옛 이름을 그대로**\n#:    둡니다 (2026-09-10 에 4군 이름만 보호자 말로 바꿨습니다).\nMORPH_GROUP: dict[str, str] = {\n    \"A1\": \"융기·발진\",   \"A4\": \"융기·발진\",      # 둘 다 작은 융기, 고름 유무로만 갈림\n    \"A2\": \"표면 변화\",   \"A3\": \"표면 변화\",      # 표피 각질 / 색·두께 변화\n    \"A5\": \"손상·덩어리\", \"A6\": \"손상·덩어리\",    # 조기 진료 권장 쪽\n}\n\n#: ★ 같은 묶음인데 **A6(결절·종괴)만 따로 둡니다.** A6 은 종양 감별이 필요한\n#: 유일한 클래스라, 다른 것과 한 이름으로 부르면 그 뜻이 사라집니다.\n#: 실측(STEP 28): 커버리지 62.1% → **61.4%** 로 **0.7%p 밖에 안 잃습니다.**\n#: 전체 데이터 정확도도 0.8365 → 0.8315.\nURGENT_CODES: tuple[str, ...] = (\"A5\", \"A6\")\n\"\"\"★ 계열 4군 중 **급한 쪽의 클래스 코드**. 이름이 아니라 코드로 적습니다.\n\n⚠️ **예전에는 이름 문자열이었습니다** (`(\"미란·궤양\", \"결절·종괴\")`).\n그러면 `MORPH_GROUP_KEEP_A6` 의 이름을 바꿀 때 여기를 같이 안 고치면\n**하향 방지 규칙이 조용히 안 걸립니다** — 에러도 안 나고, A5 하향이\n36.8% 에서 43.8% 로 돌아갑니다. 2026-09-10 에 이름을 바꾸면서 코드로\n옮겼습니다. `URGENT_GROUPS` 는 이제 **아래에서 파생**됩니다.\"\"\"\n\nDOWNGRADE_BLOCK_MIN = 0.25\n\"\"\"★ **하향 방지 규칙**의 문턱 (STEP 35).\n\n    급한 쪽(`URGENT_GROUPS`) 확률의 **합**이 이 값 이상이면,\n    덜 급한 묶음은 **답 후보에서 뺍니다.**\n\n그러면 급한 쪽 이름을 말하거나, 확신이 모자라면 **아무 말도 안 합니다.**\n\n왜 필요한가 — 전체 하향은 3.7% 로 관문(5%)을 통과하는데 **말한 미란·궤양의\n43.8%** 가 하향이었습니다. 표면 변화(전체의 51%)가 평균을 떠받쳐 A5 가 묻힙니다.\n\n실측(holdout): 커버리지 67.9 → **66.5%**(−1.3%p), A5 하향 43.8 → **36.8%**\n(−7.0%p), 전체 하향 3.7 → 2.9%. **추론 0회**이고 릴리스 단독보다 두 축 다 낫습니다.\n\n⚠️ 문턱은 **val 에서 고르고 holdout 은 확인만** 했습니다 (통과한 건 0.25 하나).\n\n★ 이 값은 **판단으로 번역됩니다** — `0.25 ≈ \"하향은 다른 오답의 2배로 나쁘다\"`.\n3배로 보면 커버리지 62.7%, 5배면 55.0%, 8배면 39.8% 입니다. *\"0.25 로 하자\"* 는\n반박 불가지만 *\"하향은 과잉보다 2배 나쁘다\"* 는 **반박 가능**합니다 — 그게 이\n상수가 있는 자리입니다. 재현: `uv run python tools/a5_downgrade.py`\"\"\"\n\nA6_ALERT_MIN = 0.40\n\"\"\"★ **\"덩어리가 의심됩니다\" 경보**의 문턱 (STEP 34, 2026-09-08 켜기로 결정).\n\n⚠️ **`p(이상) × p(A6)` 에 겁니다** — `p(A6)` 단독이 아닙니다.\n`tools/naming_granularity.py` 의 `score = p1 * ens[:, ia6]` 와 같은 값이어야\nSTEP 34 의 표를 그대로 읽을 수 있습니다. `GROUP_CONF_MIN` 과 같은 모양입니다.\n\n실측 (STEP 34 — **같은 문턱**에서 val / holdout):\n\n    문턱    val 재현율   ho 재현율   val 정밀도   ho 정밀도\n    0.30     68.6%      68.2%       71.6%      58.2%\n    0.40     61.1%      59.4%       81.4%      71.3%      <- 채택\n    0.50     55.0%      51.8%       84.0%      77.6%\n    0.70     38.4%      36.9%       90.4%      85.1%\n\n★ **정밀도가 아니라 재현율로 정했습니다.** 정밀도는 모델의 성질이 아니라\n**모델 × 유병률**의 성질이라 문턱을 고정해도 안 고정됩니다 (A6 이 holdout 에서\n절반만큼 드물어 정밀도만 5~13%p 떨어졌습니다). 재현율은 유병률과 무관해\n양쪽에서 유지됩니다 — 0.40 에서 **약 60%**.\n\n⚠️ 이 경보만 **병변 이름을 말합니다.** 예외인 이유:\n  · 임상 해설이 *\"A6 으로 오탐하는 건 상대적으로 안전\"* 이라고 적어뒀습니다\n    (병원에 가서 확인하면 되니까) — 반대 방향(놓침)이 훨씬 나쁩니다\n  · A6 은 **종양 감별**이 필요한 유일한 클래스라 4묶음에서도 혼자 뒀습니다\n끄려면 `DOG_SKIN_SHOW_A6_ALERT=0`.\"\"\"\n\nMORPH_GROUP_KEEP_A6: dict[str, str] = {\n    \"A1\": \"솟아오른 변화\", \"A4\": \"솟아오른 변화\",\n    \"A2\": \"피부 표면·색·두께 변화\", \"A3\": \"피부 표면·색·두께 변화\",\n    \"A5\": \"벗겨지거나 패인 상처\", \"A6\": \"깊거나 단단한 혹\",\n}\n\n#: ★ 급한 쪽 **묶음 이름** — `URGENT_CODES` 에서 **파생**합니다.\n#:   손으로 적지 마세요. 위 표의 이름을 바꾸면 여기가 알아서 따라옵니다.\nURGENT_GROUPS: tuple[str, ...] = tuple(\n    dict.fromkeys(MORPH_GROUP_KEEP_A6[c] for c in URGENT_CODES))\n\n#: ★ **그 묶음이 어떤 라벨을 담고 있나** — 화면에 괄호로 붙습니다 (2026-09-10).\n#:\n#:   `솟아오른 변화` 만 들고 병원에 가면 **수의사가 못 알아듣습니다.** 보호자가\n#:   전달할 수 있는 말이 있어야 합니다. 이건 **데이터 라벨의 이름**입니다.\n#:\n#: ⚠️ **\"1등 병변\" 과 다릅니다.** 그건 *\"이 개는 구진입니다\"* 라고 **단정**하는\n#:    것이고(holdout 46.3% 틀림), 이건 *\"이 묶음은 이런 것들을 담는다\"* 는\n#:    **용어 풀이**입니다. 예측이 아니라 정확도 문제가 안 걸립니다.\n#: ⚠️ **순서는 코드순(A1→A6)으로 고정합니다.** 확률순으로 두면 첫 이름이\n#:    \"1등\" 으로 읽혀서 그때는 진짜로 top1 을 되살리는 셈이 됩니다.\nGROUP_LABELS: dict[str, str] = {\n    g: \"·\".join(CLASS_KO[c] for c in CLASSES\n                if c != NORMAL_LABEL and MORPH_GROUP_KEEP_A6[c] == g)\n    for g in dict.fromkeys(MORPH_GROUP_KEEP_A6.values())\n}\n\n#: ★ **보호자가 사진에서 보는 특징** (2026-09-10, 수의학 근거 대조 후 채택).\n#:   화면에 계열 이름 바로 아래 한 줄로 붙습니다.\n#:   ⚠️ *\"피가 나면 신속 진료\"* 같은 **조건부 긴급도를 넣지 마세요** — STEP 30 에서\n#:      *\"계열 이름과 긴급도를 같이 띄우면 말한 것의 절반이 한 단계 부풀려진다\"* 로\n#:      막아둔 자리입니다. 진료 권고는 이미 마지막 줄에서 한 번 나갑니다.\nGROUP_FEATURE: dict[str, str] = {\n    #   ⚠️ 받은 표는 \"고름이 찬 **병변**\" 이었는데, 보호자 화면에서 `병변` 을 빼기로\n    #      해서(2026-09-10) 여기만 \"자리\" 로 바꿨습니다. 되돌리려면 이 줄만 고치면 됩니다.\n    \"솟아오른 변화\":           \"돌기, 넓게 솟은 부위, 고름이 찬 자리\",\n    \"깊거나 단단한 혹\":        \"피부 안쪽 또는 표면의 덩어리\",\n    \"피부 표면·색·두께 변화\":  \"딱지, 둥근 비늘, 검어진 피부, 두꺼워진 피부\",\n    \"벗겨지거나 패인 상처\":    \"까짐, 진물, 출혈, 깊게 패인 부위\",\n}\n\n#: ★ **수의학적 의미** — \"자세히 보기\" 에만 넣습니다. 본문에 두지 않습니다.\n#:   primary/secondary 는 *진단 순서*의 축이지 *보호자에게 뭐라고 부를지*의\n#:   축이 아닙니다 (STEP 32 에서 그 축으로 2군을 만들었다가 과잉 88.4% 로 기각).\nGROUP_DETAIL: dict[str, str] = {\n    \"솟아오른 변화\":           \"주로 일차 병변. 단단한 병변과 내용물이 찬 병변은 하위 분류로 구분\",\n    \"깊거나 단단한 혹\":        \"염증·낭종·종양 등을 감별해야 하는 일차 병변\",\n    \"피부 표면·색·두께 변화\":  \"병변 진행 후 흔적 또는 만성 염증성 변화\",\n    \"벗겨지거나 패인 상처\":    \"표피 또는 더 깊은 조직이 소실된 이차 병변\",\n}\n\n#: ★ **문헌 근거 (2026-09-08, 원문을 직접 열어 확인)** — 세 축이 각각 표준입니다:\n#:   ① primary / secondary — Merck Vet Manual 이 목록으로 나눕니다\n#:      (secondary: epidermal collarettes · erosions/ulcers · lichenification …)\n#:   ② **1cm 경계** → **A6 을 따로 둔 근거**\n#:      *\"solid elevated lesion **<1cm** diameter\"* (papule) /\n#:      *\"circumscribed solid elevation **>1cm** in diameter that usually\n#:       **extends into deeper layers of skin**\"* (nodule) — Cornell AHDC.\n#:      Veterian Key 도 *\"approximately 1 cm in diameter or smaller\"*.\n#:      ⚠️ **Merck 페이지 자체에는 cm 수치가 없습니다** — 여기 근거로 대지 마세요.\n#:   ③ **표피 소실 여부** → **A5 를 가르는 선**\n#:      Merck: 미란·궤양은 *\"**loss of the epidermis**\"*.\n#:      ⚠️ **\"기저막(basement membrane) 파괴 여부\" 로 말하면 틀립니다.**\n#:         미란(erosion)은 표피 일부만 잃고 **기저막은 온전**하며 흉터 없이\n#:         낫습니다. 기저막까지 가는 것은 궤양(ulcer)뿐입니다. 그 축으로\n#:         가르면 **미란이 A2·A3 쪽에 붙어** 묶음이 무너집니다.\n#:         우리 축은 *표피가 쌓이는가(A2·A3) / 소실되는가(A5)* 입니다.\n#:      장벽이 깨지면 `S. pseudintermedius` 2차 감염 위험 —\n#:      Hillier et al. (2014) *Vet Dermatol* 25(3):163-e43 (ISCAID 지침)\n#:   그리고 A2+A3 는 **Hensel, Santoro, Favrot, Hill, Griffin (2015),\n#:   BMC Vet Res**(ICADA 개 아토피 가이드라인)의 한 문장에 같이 있습니다:\n#:      \"Typical secondary skin lesions are excoriations, alopecia,\n#:       **lichenification, hyperpigmentation, crusting, and seborrhea**.\"\n#: ⚠️ 단 그 목록엔 `excoriations` 도 들어 있는데 우리는 그걸 A5 쪽으로 가릅니다.\n#:    즉 그 문장 하나로 정해지지 않고 **②③ 축과의 조합**이 우리 묶음입니다.\n#:    **이 조합을 쓴 선례는 못 찾았습니다.**\n#:\n#: ⚠️ **정직하게 적어둡니다** — 임상 해설이 \"안전한 혼동\" 으로 **명시한 것은\n#: 두 쌍뿐**입니다 (A1↔A4 · A5↔A6). `MORPH_GROUP` 의 A2+A3 은 그 문서가 인정한\n#: 게 아니라 **남은 것**이고, 이득의 대부분이 거기서 나옵니다:\n#:     6종 33.7% → 문서가 인정한 병합만 42.5% → A2+A3 까지 62.1%\n#: 즉 **\"임상적으로 비슷해서 묶었다\" 는 절반만 맞습니다.**\nDOC_ENDORSED_MERGES: tuple[tuple[str, str], ...] = ((\"A1\", \"A4\"), (\"A5\", \"A6\"))\n\n#: ★ 수의피부과 **표준 축** — primary(병이 직접 만든 것) vs secondary(그 뒤에\n#: 생긴 것). 우리가 만든 묶음이 아니라 교과서 분류입니다:\n#:   primary    macule · papule · plaque · wheal · vesicle · **pustule** · **nodule**\n#:   secondary  **collarette** · scar · excoriation · **erosion** · **ulcer** ·\n#:              fissure · **lichenification** · callus\n#:   둘 다 됨    **scale · crust · 색소 변화** (원인에 따라)\n#: 출처: MSD Veterinary Manual · Clinicians Brief (2026-09-06 확인)\n#:\n#: ⚠️ **A2·A3 는 깨끗이 안 갈립니다** — A2 의 잔고리는 secondary 인데 비듬·가피는\n#:    '둘 다', A3 의 태선화는 secondary 인데 색소침착은 '둘 다' 입니다.\n#:    ★ 그런데 **둘이 똑같이 애매합니다.** 출처가 *\"secondary 는 여러 primary 에서\n#:    나올 수 있어 진단 가치가 낮다\"* 고 적는데, A2+A3 를 한 통에 넣는다는 건\n#:    **덜 진단적인 둘을 묶는 것**이라 사후 정당화가 아닌 근거가 됩니다.\n#: ⚠️ A6 를 따로 둔 근거인 **1cm 는 교과서 경계**입니다 (papule ≤1cm /\n#:    nodule >1cm, 결절은 진피·피하로 더 깊이) — 표준과 맞습니다.\n#:    🔴 **단 \"A1·A4 = ≤1cm\" 로 말하면 틀립니다 (2026-09-10 정정).**\n#:    A1 의 **플라크(plaque)는 보통 >1cm** 입니다. 1cm 는 *구진 vs 결절* 의\n#:    경계이지 *묶음 전체*의 크기가 아닙니다. 융기·발진의 축은 크기가 아니라\n#:    **\"표면보다 솟았는가\"** 이고, A1(단단함)과 A4(고름이 참)는 그 안에서\n#:    내용물로 갈립니다.\n#: 🚫 **그런데 이 축으로 2군을 만들면 기각입니다** (STEP 32): 커버리지는 제일\n#:    높은데(74.6%) primary 에 A6 이 들어 있어 **과잉 분류가 88.4%** 입니다 —\n#:    구진 하나에도 '조기 진료' 가 붙어 그 말이 뜻을 잃습니다.\nLESION_ORIGIN: dict[str, str] = {\n    \"A1\": \"primary\", \"A4\": \"primary\", \"A6\": \"primary\",\n    \"A5\": \"secondary\",\n    \"A2\": \"mixed\",   # 잔고리 secondary / 비듬·가피 '둘 다'\n    \"A3\": \"mixed\",   # 태선화 secondary / 색소침착 '둘 다'\n}\n\n#: 🔴 위험한 혼동 — 임상 해설에 적힌 그대로. **긴급도를 낮춰 말하는 것**입니다.\n#:    (A6→A2 가 최악, A5→A1, A6→A1). 일반화하면 \"실제 등급 > 말한 등급\".\nDANGEROUS_IS_UNDER_TRIAGE = True\n\n# ──────────────────────────────────────────────────────────────\n# 촬영 가이드 밴드 — **이 값들의 출처는 여기 하나뿐입니다**\n# ──────────────────────────────────────────────────────────────\n# `robust.usable_range()` 가 STEP 16 에서 실측한 배율 밴드입니다 (n=2,000).\n# 쓰는 곳:\n#     src/agent.py        화면 점유율로 (÷ 2.5 — 2단계 크롭이 m2.5)\n#     tools/box_error.py  네모 크기 오차로 뒤집어서 (1 / 배율)\n#     demo/index.html     화면\n#     docs/SERVING.md     사람이 읽는 판\n#\n# ⚠️ **왜 `robust.py` 가 아니라 여기인가** — `robust.py` 는 torch 를 import\n#    합니다. 밴드만 필요한 쪽(`box_error.py` 는 `uv run python` 으로 도는\n#    측정 도구)이 torch 를 끌어오게 됩니다. 실제로 그래서 백엔드 사본이\n#    깨졌습니다 (2026-09-06). **상수는 의존성 없는 곳에 둡니다.**\n#\n# ⚠️ STEP 10 은 (0.85, 1.4) / (0.7, 1.7) 이었습니다. **크게 찍는 쪽이\n#    빡빡해졌습니다** — 2.0x 에서 macro-F1 이 0.449 까지 떨어집니다.\n# ⚠️ 데이터가 늘거나 크롭이 바뀌면 `usable_range()` 를 **다시 돌려** 고치세요.\nZOOM_RECOMMEND = (0.7, 1.2)     # 최고점 대비 하락 5% 이내 (STEP 16, n=2,000)\nZOOM_ALLOW = (0.6, 1.4)         # 하락 10% 이내\nZOOM_CENTER_MAX = 0.10          # 병변이 화면 중앙에서 이만큼 이내\n\n# AI Hub 데이터셋 식별자\nAIHUB_DATASET_KEY = \"561\"\nAIHUB_DATASET_NAME = \"반려동물 피부 질환 데이터\"\n\n# 우리가 쓸 데이터 범위 (사용자 결정: 반려견 + 일반카메라만)\nINCLUDE_SPECIES = [\"반려견\"]\nEXCLUDE_SPECIES = [\"반려묘\"]\nINCLUDE_CAMERA = [\"일반카메라\"]\nEXCLUDE_CAMERA = [\"더모스코프\"]  # 보호자가 만들 수 없는 입력이므로 제외\n\n\n# ──────────────────────────────────────────────────────────────\n# 실행 설정\n# ──────────────────────────────────────────────────────────────\n@dataclass\nclass CFG:\n    # --- 재현성 ---\n    seed: int = 42\n    deterministic: bool = False\n\n    # --- 데이터 ---\n    img_size: int = 288\n    crop_margin: float = 1.5          # ROI 크롭 여유 배율. 1.0=박스 딱 맞게, 2.0=주변 2배\n    crop_min_px: int = 64             # 이보다 작은 병변 박스는 버림 (노이즈)\n    # >0 이면 margin 대신 **고정 픽셀 창**으로 자릅니다 (병변 중심, 항상 같은 크기).\n    # margin 크롭은 병변 크기에 따라 확대 배율이 달라져 그 배율이 정답을 흘립니다.\n    # → src/crop.py 의 fixed_box() 설명, docs/cautions/08 참고\n    crop_fixed_px: int = 0\n    save_crop_size: int = 512         # 디스크에 저장할 크롭 해상도 (학습 시 img_size 로 리사이즈)\n    save_crop_quality: int = 92\n\n    # --- 중복 제거 ---\n    phash_size: int = 16              # phash 비트 크기 (16 → 256bit, 기본 8보다 정밀)\n    dedup_hamming: int = 6            # 이 거리 이하면 near-duplicate 로 봄\n\n    # --- 분할 ---\n    n_folds: int = 5\n    use_fold: int = 0                 # 단일 실험에서 검증에 쓸 fold\n    holdout_ratio: float = 0.15       # 최종 1회만 보는 테스트셋 비율 (개체 단위)\n\n    # --- 모델 ---\n    model_name: str = \"tf_efficientnetv2_s.in21k_ft_in1k\"\n    pretrained: bool = True\n    drop_rate: float = 0.2\n    drop_path_rate: float = 0.1\n\n    # --- 학습 ---\n    epochs: int = 15\n    batch_size: int = 0               # 0 이면 env.suggest_batch_size() 로 자동\n    grad_accum: int = 1\n    lr: float = 3e-4\n    backbone_lr_mult: float = 0.1     # 백본은 헤드보다 낮은 lr (파인튜닝 관례)\n    weight_decay: float = 0.05\n    warmup_epochs: int = 2\n    label_smoothing: float = 0.1\n    amp: bool = True\n    ema_decay: float = 0.999          # 0 이면 EMA 끔\n    clip_grad_norm: float = 1.0\n    num_workers: int = -1        # -1 = CPU 코어 수에 맞춰 자동\n    early_stop_patience: int = 5\n    monitor: str = \"macro_f1\"         # ⚠️ accuracy 아님. 불균형 데이터에서 accuracy 는 거짓말을 합니다.\n\n    # --- 증강 ---\n    # ⚠️ 피부 병변은 \"색과 질감\"이 곧 라벨입니다.\n    #    강한 색상 증강은 A3(과다색소침착)을 A1 처럼 만들어 라벨을 파괴합니다.\n    #    아래 값은 일반 이미지 분류 기본값보다 의도적으로 약하게 잡았습니다.\n    rrc_scale: tuple[float, float] = (0.7, 1.0)\n    # ⚠️ RandomResizedCrop 은 **축소를 못 합니다.** 이미지의 일부를 잘라 확대할 뿐이라\n    #    가장 축소된 경우가 \"이미지 전체\"(검증 대비 약 0.88배)이고, rrc_scale 하한을\n    #    낮추면 **확대 쪽만** 넓어집니다. 실측:\n    #        default(0.70,1.0)      → 0.88x ~ 1.05x\n    #        scale_robust(0.35,1.0) → 0.88x ~ 1.49x\n    #    그런데 배율 교란 검사는 0.5x·0.71x 를 묻습니다. 훈련에서 **한 번도 안 본**\n    #    구간이라, rrc_scale 을 아무리 넓혀도 그 하락은 안 줄어듭니다 (실측 확인).\n    #    축소를 배우려면 이미지를 줄여 여백을 채우는 affine 변환이 필요합니다.\n    affine_scale: tuple[float, float] | None = None   # 예: (0.5, 1.3) — 축소 포함\n    hflip: float = 0.5\n    vflip: float = 0.0                # 피부 사진은 위아래 뒤집기가 부자연스러움\n    rotate_deg: int = 15\n    color_jitter: float = 0.1         # brightness/contrast/saturation 공통 강도 (약하게!)\n    hue_jitter: float = 0.02          # 색조는 특히 조심 — 거의 건드리지 않음\n    randaugment_n: int = 0            # 0 이면 끔. 켜려면 2 권장\n    randaugment_m: int = 7\n    mixup_alpha: float = 0.0          # 실험용. 켜면 0.2 권장\n    cutmix_alpha: float = 0.0\n    random_erasing: float = 0.15\n\n    # --- albumentations 전용 노브 ---\n    # torchvision 에는 없거나 느린 것들. albumentations 가 없으면 전부 무시됩니다.\n    # ⚠️ 촬영 조건(흐림·노이즈·조명)을 흉내 내는 쪽입니다. 실측: 정상 사진의\n    #    선명도 중앙값이 39, 병변은 271 — 모델이 화질로 맞힐 여지가 있어서\n    #    흐림을 훈련에 넣어두면 그 지름길을 막는 효과도 기대할 수 있습니다.\n    blur_p: float = 0.0               # 가우시안/모션 블러 확률\n    noise_p: float = 0.0              # 센서 노이즈 확률\n    jpeg_p: float = 0.0               # JPEG 압축 열화 (보호자 사진은 대개 압축됨)\n    clahe_p: float = 0.0              # 국소 대비 보정 — 질감을 살리는 방향\n    shift_limit: float = 0.0          # 평행이동 비율 (위치 교란 20.6% 대응)\n\n    # --- 불균형 대응 ---\n    balance_strategy: str = \"class_weight\"\n    # \"none\" | \"class_weight\" | \"weighted_sampler\" | \"hair_weighted\"\n    #   hair_weighted — 털처럼 가는 선이 많은 **정상** 사진을 더 자주 뽑습니다.\n    #   헛알림 실측(AUROC 0.749)에서 나온 값이고, 클래스 총량은 보존합니다.\n    hair_alpha: float = 1.0   # 0=끔, 1=최상위가 최하위보다 2배 자주\n    focal_gamma: float = 0.0                 # >0 이면 focal loss 사용\n\n    # --- 평가 / 안전장치 ---\n    tta_hflip: bool = True\n    calibrate: bool = True                   # 온도 스케일링\n    target_recall_stage1: float = 0.95       # 1단계 정상/이상: 놓치지 않는 게 우선\n    abstain_threshold: float = 0.45          # 최고확률이 이 미만이면 \"판단 어려움\"\n    topk_report: int = 3                     # 사용자에게 상위 몇 개까지 보여줄지\n\n    # --- 로깅 ---\n    exp_name: str = \"baseline\"\n    log_every: int = 50\n    use_wandb: bool = False\n\n    def resolved_batch_size(self) -> int:\n        if self.batch_size > 0:\n            return self.batch_size\n        from src import env\n\n        scale = _infer_scale(self.model_name)\n        # 백본별 메모리 보정 — timm 이름으로 MODEL_ZOO 를 되짚습니다\n        mf = 1.0\n        for spec in MODEL_ZOO:\n            if spec.timm_name == self.model_name or spec.key == self.model_name:\n                mf = spec.mem_factor\n                break\n        return env.suggest_batch_size(self.img_size, scale, mem_factor=mf)\n\n    def resolved_num_workers(self) -> int:\n        \"\"\"-1 이면 CPU 코어 수에 맞춰 정합니다.\"\"\"\n        if self.num_workers >= 0:\n            return self.num_workers\n        from src import env\n\n        return env.suggest_workers()\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n    def save(self, path: str | Path) -> Path:\n        import json\n\n        p = Path(path)\n        p.parent.mkdir(parents=True, exist_ok=True)\n        p.write_text(json.dumps(self.to_dict(), indent=2, ensure_ascii=False), encoding=\"utf-8\")\n        return p\n\n    @classmethod\n    def from_dict(cls, data: dict[str, Any]) -> \"CFG\":\n        \"\"\"알 수 없는 키는 무시하고 만듭니다 (설정 파일이 구버전이어도 동작).\"\"\"\n        data = dict(data)\n        # tuple 필드 복원 (JSON 은 list 로 저장됨)\n        if isinstance(data.get(\"rrc_scale\"), list):\n            data[\"rrc_scale\"] = tuple(data[\"rrc_scale\"])\n        if isinstance(data.get(\"affine_scale\"), list):\n            data[\"affine_scale\"] = tuple(data[\"affine_scale\"])\n        known = set(cls.__dataclass_fields__)\n        return cls(**{k: v for k, v in data.items() if k in known})\n\n    @classmethod\n    def load(cls, path: str | Path) -> \"CFG\":\n        import json\n\n        return cls.from_dict(json.loads(Path(path).read_text(encoding=\"utf-8\")))\n\n\ndef _infer_scale(model_name: str) -> str:\n    n = model_name.lower()\n    for tag in (\"nano\", \"tiny\", \"small\", \"base\", \"large\", \"huge\"):\n        if tag in n:\n            return {\"nano\": \"tiny\", \"huge\": \"large\"}.get(tag, tag)\n    # efficientnet_b0..b7 같은 이름 처리\n    if \"_b0\" in n or \"_b1\" in n or \"_s.\" in n or n.endswith(\"_s\"):\n        return \"small\"\n    if \"_b2\" in n or \"_b3\" in n or \"_m.\" in n:\n        return \"base\"\n    return \"base\"\n\n\n# ──────────────────────────────────────────────────────────────\n# 증강 프리셋\n#\n# 기본값은 의도적으로 약합니다 — 피부 병변은 색과 질감이 곧 라벨이라\n# 강한 색상 증강은 A3(과다색소침착)를 A1 처럼 만들어 버립니다.\n#\n# 그런데 **배율**은 사정이 다릅니다. 크롭이 병변 박스에 맞춰 잘리기 때문에\n# 배율이 클래스마다 다르고(실측 A1 0.47% ~ A6 3.08%, 6.5배), 모델이 그걸\n# 단서로 쓸 수 있습니다. 실사용에서 배율은 무작위이므로 그건 무너집니다.\n#\n# ⚠️ 기본 rrc_scale=(0.7, 1.0) 은 면적 1.43배 범위(선형 1.2배)입니다.\n#    막아야 할 격차가 선형 2.5배인데 이건 아무 효과가 없습니다.\n#\n# ⚠️ 넓은 배율 증강에는 **여유가 있는 크롭**이 필요합니다.\n#    m1.5 처럼 딱 붙은 크롭에 0.15 를 걸면 병변이 화면에서 잘려 나가\n#    라벨이 깨집니다. m2.5 나 f512 같은 넉넉한 크롭을 base 로 쓰세요.\n#    효과 판정은 src/robust.py 의 scale_stress() 로 합니다.\n# ──────────────────────────────────────────────────────────────\nAUG_PRESETS: dict[str, dict] = {\n    # ── 기준 ────────────────────────────────────────────────────\n    \"default\": {},\n\n    # ── ① 폭을 **줄이는** 쪽 (멘토 피드백 1번) ──────────────────\n    # 지금까지 넓히기만 두 번 시도해 둘 다 실패했습니다. 반대 방향은 안 해봤습니다.\n    # 좁히면 과제가 쉬워져 점수가 오를 수 있고, 대신 견고성은 나빠질 수 있습니다.\n    # 좁히기 축에 **점을 두 개** 찍습니다. 하나만 찍으면 \"이겼다\" 는 알아도\n    # \"더 좁혀야 하나\" 를 몰라서 또 한 판 돌려야 합니다.\n    #     0.70(default) → 0.85(narrow) → 0.92(narrower)\n    #   계속 좋아지면 더 좁히고, narrow 가 최고면 거기가 최적점이고,\n    #   셋이 비슷하면 이 축은 상관없다는 뜻입니다. 한 번에 결론이 납니다.\n    \"narrow\": {\"rrc_scale\": (0.85, 1.0), \"rotate_deg\": 10, \"random_erasing\": 0.0},\n    \"narrower\": {\"rrc_scale\": (0.92, 1.0), \"rotate_deg\": 5, \"random_erasing\": 0.0},\n\n    # ── ② 축소를 가르치는 쪽 (2단계 최악 조건이 0.5x) ───────────\n    # affine 이 이미지를 실제로 줄이고 여백을 채웁니다. RRC 는 축소를 못 합니다.\n    \"zoom_both\": {                       # 세게 — 224px 에서는 실패했던 설정\n        \"rrc_scale\": (0.5, 1.0),\n        \"affine_scale\": (0.45, 1.25),\n        \"rotate_deg\": 20,\n        \"random_erasing\": 0.25,\n    },\n    \"zoom_mild\": {                       # 완만하게 — ①과 ②의 절충\n        \"rrc_scale\": (0.75, 1.0),\n        \"affine_scale\": (0.70, 1.15),\n        \"rotate_deg\": 12,\n    },\n\n    # ── ③ 위치 교란 대응 (실측 20.6% 하락) ─────────────────────\n    \"shift\": {\"shift_limit\": 0.15, \"rotate_deg\": 15},\n\n    # ── ④ 촬영 조건 흉내 (멘토 피드백 4번) ─────────────────────\n    # 배율은 기본값 그대로 두고 화질만 흔듭니다.\n    # 실측: 정상 사진 선명도 중앙값 39 vs 병변 271 — 모델이 화질로 맞힐 여지가\n    # 있어서, 흐림을 훈련에 넣으면 그 지름길을 막는 효과도 기대할 수 있습니다.\n    \"photometric\": {\"blur_p\": 0.3, \"noise_p\": 0.25, \"jpeg_p\": 0.3, \"clahe_p\": 0.2},\n\n    # ── ⑤ 조합 ─────────────────────────────────────────────────\n    # ⚠️ 아래 둘은 정의만 남겨둡니다 — 03b 스윕에서는 뺐습니다.\n    #    zoom_shift 는 zoom_mild + shift 결과로 대충 예상되고,\n    #    kitchen_sink 는 \"너무 세면 나빠진다\" 를 확인하는 대조군이라 기대값이 낮습니다.\n    \"zoom_shift\": {                      # 축소 + 이동 (두 교란을 같이)\n        \"rrc_scale\": (0.75, 1.0),\n        \"affine_scale\": (0.70, 1.15),\n        \"shift_limit\": 0.12,\n        \"rotate_deg\": 15,\n    },\n    \"kitchen_sink\": {                    # 전부 — 과한 게 해로운지 확인하는 상한선\n        \"rrc_scale\": (0.6, 1.0),\n        \"affine_scale\": (0.55, 1.25),\n        \"shift_limit\": 0.15,\n        \"rotate_deg\": 20,\n        \"blur_p\": 0.25, \"noise_p\": 0.2, \"jpeg_p\": 0.25,\n        \"random_erasing\": 0.25,\n    },\n\n    # ── 과거 실험용 (결론 남음. 참고로만 둡니다) ────────────────\n    # 확대만 넓힙니다. 2단계 최악 조건이 축소라 여기엔 효과가 없습니다.\n    \"scale_robust\": {\"rrc_scale\": (0.35, 1.0), \"rotate_deg\": 20, \"random_erasing\": 0.25},\n}\n\n\n# 파인튜닝 강도 프리셋\n#\n# ⚠️ 용어 정리 — 자주 헷갈립니다.\n#   \"전이학습(transfer learning)\" 은 우산 개념입니다. 그 안에 두 방식이 있습니다:\n#     · linear probe   백본을 얼리고 헤드만 학습 (freeze_backbone(True))\n#     · fine-tuning    백본까지 같이 학습 ← **우리는 처음부터 이쪽입니다**\n#   `freeze_backbone()` 은 정의만 되어 있고 어디서도 호출하지 않습니다.\n#\n# 그러면 남는 질문은 \"파인튜닝을 할까?\" 가 아니라 \"얼마나 세게 할까?\" 입니다.\n# 그걸 정하는 게 backbone_lr_mult 입니다:\n#\n#     헤드 lr   = cfg.lr                        (랜덤 초기화라 빨리 배워야 함)\n#     백본 lr   = cfg.lr × backbone_lr_mult     (사전학습 지식을 지키려고 낮춤)\n#\n# 기본 0.1 은 \"ImageNet 특징이 이미 쓸만하다\" 는 전제입니다. 그런데 우리 과제는\n# 물체 인식이 아니라 **피부 질감·색의 미세한 구분**이라 도메인 격차가 큽니다.\n# 백본이 3e-5 로 움직이면 12 에폭 동안 거의 제자리입니다.\n#\n# 실측 근거 (VL01 2단계): train 1.352 / val 1.474 — 학습 데이터조차 잘 못 맞춥니다.\n# 과적합이 아니라 **덜 배운** 상태이고, 백본 lr 이 유력한 원인입니다.\n# ──────────────────────────────────────────────────────────────\nFT_PRESETS: dict[str, dict] = {\n    # 지금까지 쓰던 설정 (비교 기준)\n    \"conservative\": {\"backbone_lr_mult\": 0.1},\n\n    # 백본을 3배 더 움직입니다. 도메인 격차가 클 때의 표준적인 선택.\n    \"moderate\": {\"backbone_lr_mult\": 0.3, \"warmup_epochs\": 3},\n\n    # 백본과 헤드를 같은 lr 로. 격차가 아주 클 때 가장 좋을 수 있지만\n    # 사전학습 지식을 잃을 위험(catastrophic forgetting)이 있어 warmup 을 길게 둡니다.\n    \"aggressive\": {\"backbone_lr_mult\": 1.0, \"warmup_epochs\": 4, \"lr\": 1e-4},\n\n    # 백본을 얼리고 헤드만. 우리 데이터(1.5만장)에는 부족하지만,\n    # \"백본 적응이 실제로 기여하는가\" 를 재는 대조군으로 유용합니다.\n    \"linear_probe\": {\"backbone_lr_mult\": 0.0},\n}\n\n\ndef ft_preset(name: str) -> dict:\n    \"\"\"파인튜닝 강도 프리셋 → CFG 오버라이드 사전.\"\"\"\n    if name not in FT_PRESETS:\n        raise KeyError(f\"모르는 프리셋: {name}. 가능: {sorted(FT_PRESETS)}\")\n    return dict(FT_PRESETS[name])\n\n\ndef with_finetune(cfg: \"CFG\", name: str) -> \"CFG\":\n    \"\"\"cfg 에 파인튜닝 강도 프리셋을 얹은 새 CFG.\"\"\"\n    d = {**cfg.to_dict(), **ft_preset(name)}\n    if name != \"conservative\":\n        d[\"exp_name\"] = f\"{cfg.exp_name}_{name}\"\n    return CFG.from_dict(d)\n\n\ndef aug_preset(name: str) -> dict:\n    \"\"\"프리셋 이름 → CFG 오버라이드 사전.\n\n        cfg = CFG(**{**CFG(model_name=\"resnet50\").to_dict(), **aug_preset(\"scale_robust\")})\n    \"\"\"\n    if name not in AUG_PRESETS:\n        raise KeyError(f\"모르는 프리셋: {name}. 가능: {sorted(AUG_PRESETS)}\")\n    return dict(AUG_PRESETS[name])\n\n\ndef with_aug(cfg: \"CFG\", name: str) -> \"CFG\":\n    \"\"\"cfg 에 증강 프리셋을 얹은 새 CFG. 실험 이름에 프리셋을 붙여 둡니다.\"\"\"\n    over = aug_preset(name)\n    d = {**cfg.to_dict(), **over}\n    if name != \"default\":\n        d[\"exp_name\"] = f\"{cfg.exp_name}_{name}\"\n    return CFG.from_dict(d)\n\n\n# ──────────────────────────────────────────────────────────────\n# 노트북 버전\n# ──────────────────────────────────────────────────────────────\n# ⚠️ 노트북 셀은 `git pull` 로 갱신되지 않습니다. Colab/Kaggle 에 올린 .ipynb 는\n#    다시 import 하기 전까지 그대로입니다. src/ 만 매번 최신이 됩니다.\n#    그래서 셀을 고칠 때마다 이 값을 올리고, 노트북 첫 셀이 자기가 들고 있는\n#    값과 비교해 **낡았으면 바로 알립니다.** (몇 시간 뒤에 알게 되면 늦습니다)\nNOTEBOOK_VERSION = \"2026-09-04.5\"\n\n# ★ 채택된 2단계 크롭. STEP 4C(비교) → 4D(재기준선) 에서 확정했습니다.\n#   노트북 05 가 불러온 체크포인트의 크롭이 이것과 다르면 **멈춥니다** —\n#   실제로 예전 실행(m1.5)의 출력을 붙이고 그대로 진행할 뻔했습니다.\n#   그러면 촬영 가이드·보정·임계값이 전부 버린 설정 기준으로 나옵니다.\nADOPTED_STAGE2_CROP = \"m2.5\"\n\n# ──────────────────────────────────────────────────────────────\n# 모델 라인업 — STEP 4 에서 순서대로 돌립니다.\n#\n# timm 모델명은 버전마다 바뀝니다. src/models.py 가 실행 시점에\n# timm.list_models() 로 존재를 검증하고, 없으면 fallback 을 씁니다.\n# ──────────────────────────────────────────────────────────────\n@dataclass\nclass ModelSpec:\n    key: str\n    timm_name: str\n    fallbacks: list[str] = field(default_factory=list)\n    img_size: int = 288\n    scale: str = \"base\"\n    # 배치 추천을 몇 배로 줄일지. env.suggest_batch_size 의 공식이 **ResNet 기준**\n    # 이라 트랜스포머 계열의 활성값 메모리를 모릅니다. 실측으로 정했습니다:\n    #   T4(14.6GB) · 256px · 배치 32 에서\n    #     swinv2_base   → OOM\n    #     siglip2_base  → 14.3GB (98%, EMA 붙으면 터짐)\n    #   같은 조건의 convnextv2_base 는 384px 배치 12 에서 8.9GB 로 멀쩡했습니다.\n    # 그래서 어텐션 계열은 0.4 로 잡습니다 (32 → 12).\n    mem_factor: float = 1.0\n    note: str = \"\"\n\n\nMODEL_ZOO: list[ModelSpec] = [\n    ModelSpec(\n        key=\"resnet50\",\n        timm_name=\"resnet50.a1_in1k\",\n        fallbacks=[\"resnet50\"],\n        img_size=224, scale=\"base\",\n        note=\"기준선. 다른 모든 수치는 이것과 비교해서 읽습니다.\",\n    ),\n    ModelSpec(\n        key=\"effnetv2_s\",\n        timm_name=\"tf_efficientnetv2_s.in21k_ft_in1k\",\n        fallbacks=[\"tf_efficientnetv2_s\", \"efficientnet_b3\"],\n        img_size=300, scale=\"small\",\n        note=\"가볍고 강함. 모바일 배포 1순위 후보.\",\n    ),\n    ModelSpec(\n        key=\"convnextv2_base\",\n        timm_name=\"convnextv2_base.fcmae_ft_in22k_in1k\",\n        fallbacks=[\"convnextv2_tiny.fcmae_ft_in22k_in1k\", \"convnext_base.fb_in22k_ft_in1k\"],\n        img_size=288, scale=\"base\",\n        note=\"현대 CNN 최강급. 질감(texture) 표현이 좋아 피부에 잘 맞을 가능성이 큼.\",\n    ),\n    ModelSpec(\n        key=\"swinv2_base\",\n        timm_name=\"swinv2_base_window12to16_192to256.ms_in22k_ft_in1k\",\n        fallbacks=[\"swinv2_base_window8_256.ms_in1k\", \"swin_base_patch4_window7_224.ms_in22k_ft_in1k\"],\n        img_size=256, scale=\"base\",\n        mem_factor=0.4,\n        note=\"계층적 ViT. 지역 패턴 + 전역 문맥을 같이 봄.\",\n    ),\n    ModelSpec(\n        key=\"eva02_base\",\n        timm_name=\"eva02_base_patch14_448.mim_in22k_ft_in22k_in1k\",\n        fallbacks=[\"eva02_small_patch14_336.mim_in22k_ft_in1k\", \"vit_base_patch16_224.augreg2_in21k_ft_in1k\"],\n        img_size=336, scale=\"base\",\n        mem_factor=0.4,\n        note=\"정확도 상한 확인용. T4 에서는 무거우니 배치 작게 + grad_accum 사용.\",\n    ),\n    ModelSpec(\n        key=\"siglip2_base\",\n        timm_name=\"vit_base_patch16_siglip_256.v2_webli\",\n        fallbacks=[\"vit_base_patch16_siglip_224.webli\", \"vit_base_patch16_clip_224.openai\"],\n        img_size=256, scale=\"base\",\n        mem_factor=0.4,\n        note=\"대규모 이미지-텍스트 사전학습 백본. 소량 데이터에서 특히 강한 편.\",\n    ),\n]\n\nMODEL_BY_KEY = {m.key: m for m in MODEL_ZOO}\n", "src/safe_crop.py": "\"\"\"원본 사진에서 **주석된 병변을 하나도 안 자르는** random crop (STEP 44).\n\n멘토 제안 — *\"병변이 최대한 안 잘리는 범위로 random crop\"*. 저장된 크롭\n(`m2.5`·`f320`)은 이미 병변을 가운데 놓고 자른 결과라 여기서 못 씁니다.\n좌표는 **원본 JSON** 에서 옵니다 — 매니페스트의 첫 box 하나가 아닙니다.\n\ntorch 를 안 씁니다: 로컬 전처리와 학습 양쪽에서 그대로 부릅니다.\n\"\"\"\nfrom __future__ import annotations\n\nimport math\nimport random\nimport re\n\n\ndef annotation_boxes(record):\n    \"\"\"box 와 polygon 외접값을 전부 모읍니다 — 병변이 여러 개인 사진 포함.\"\"\"\n    boxes = []\n    for item in record.get(\"labelingInfo\") or []:\n        if not isinstance(item, dict):\n            continue\n        if isinstance(item.get(\"box\"), dict):\n            locations = item[\"box\"].get(\"location\")\n            if not locations:\n                raise ValueError(\"Empty box annotation\")\n            for loc in locations if isinstance(locations, list) else [locations]:\n                try:\n                    x, y, w, h = (float(loc[k]) for k in ('x', 'y', 'width', 'height'))\n                except (KeyError, TypeError, ValueError):\n                    raise ValueError(\"Malformed box annotation\")\n                boxes.append([x, y, x+w, y+h])\n        if isinstance(item.get(\"polygon\"), dict):\n            locations = item[\"polygon\"].get(\"location\")\n            if not locations:\n                raise ValueError(\"Empty polygon annotation\")\n            for loc in locations if isinstance(locations, list) else [locations]:\n                if not isinstance(loc, dict):\n                    raise ValueError(\"Malformed polygon annotation\")\n                indices = {int(k[1:]) for k in loc if re.fullmatch(r'[xy]\\d+', k)}\n                if len(indices) < 3 or indices != set(range(1, max(indices)+1)):\n                    raise ValueError(\"Malformed polygon annotation\")\n                try:\n                    pts = [(float(loc[f'x{i}']), float(loc[f'y{i}'])) for i in sorted(indices)]\n                except (KeyError, TypeError, ValueError):\n                    raise ValueError(\"Malformed polygon coordinate\")\n                if not all(math.isfinite(v) for p in pts for v in p):\n                    raise ValueError(\"Nonfinite polygon coordinate\")\n                xs, ys = zip(*pts)\n                boxes.append([min(xs), min(ys), max(xs), max(ys)])\n    return boxes\n\n\ndef sample_window(width, height, boxes, *, scale=(0.35, 1.0),\n                  ratio=(0.85, 1.18), padding=0.05, attempts=50, rng=None):\n    \"\"\"모든 box 를 여유와 함께 담는 창을 뽑습니다. `scale` 은 **원본 면적 대비 비율**.\n\n    요청한 크기·비율로 병변을 다 담을 수 없으면 **원본 전체**로 물러섭니다.\n    주석이 없는 사진도 원본 전체입니다 — 배경 아무 곳이나 자르지 않습니다.\n    좌표가 잘못됐으면 **실패시킵니다**: 조용히 잘못된 크롭으로 학습하는 것보다\n    멈추는 게 낫습니다.\n    \"\"\"\n    if width <= 0 or height <= 0:\n        raise ValueError(\"Image dimensions must be positive\")\n    if not (0 < scale[0] <= scale[1] <= 1):\n        raise ValueError(\"scale must satisfy 0 < low <= high <= 1\")\n    if not (0 < ratio[0] <= ratio[1]) or padding < 0 or attempts < 1:\n        raise ValueError(\"Invalid ratio, padding or attempts\")\n    rng = rng or random\n    full = (0, 0, width, height)\n    protected = []\n    for box in boxes:\n        if len(box) != 4 or not all(math.isfinite(v) for v in box):\n            raise ValueError(\"Invalid lesion box\")\n        x1, y1, x2, y2 = box\n        if not (0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height):\n            raise ValueError(\"Lesion box outside original image\")\n        dx, dy = (x2 - x1) * padding, (y2 - y1) * padding\n        protected.append((max(0, x1-dx), max(0, y1-dy),\n                          min(width, x2+dx), min(height, y2+dy)))\n    if not protected:\n        return full\n    left = min(b[0] for b in protected)\n    top = min(b[1] for b in protected)\n    right = max(b[2] for b in protected)\n    bottom = max(b[3] for b in protected)\n    for _ in range(attempts):\n        area = width * height * rng.uniform(*scale)\n        aspect = math.exp(rng.uniform(math.log(ratio[0]), math.log(ratio[1])))\n        w, h = round(math.sqrt(area * aspect)), round(math.sqrt(area / aspect))\n        if not (1 <= w <= width and 1 <= h <= height):\n            continue\n        xmin, xmax = max(0, math.ceil(right-w)), min(width-w, math.floor(left))\n        ymin, ymax = max(0, math.ceil(bottom-h)), min(height-h, math.floor(top))\n        if xmin <= xmax and ymin <= ymax:\n            x, y = rng.randint(xmin, xmax), rng.randint(ymin, ymax)\n            return x, y, x+w, y+h\n    return full\n\n\ndef crop_original(image, record, *, rng=None, **kwargs):\n    \"\"\"PIL 크롭과 그 원본 좌표 창을 돌려줍니다 — 리사이즈는 안 합니다.\"\"\"\n    window = sample_window(*image.size, annotation_boxes(record), rng=rng, **kwargs)\n    return image.crop(window), window\n", "src/original_data.py": "\"\"\"원본 ZIP 을 직접 읽는 학습 로더 — full / random / safe 를 같은 조건으로 (STEP 44).\n\n★ **검증은 세 방법 모두 원본 전체 letterbox 로 동일**합니다. 다른 것은 학습\n입력뿐이라, 차이가 나면 crop 방식으로 좁혀집니다.\n\n보존 crop 은 리사이즈 **전에** 원본 좌표에서 자릅니다. 그 뒤 변환은 남긴\nROI 를 자르거나 회전·이동·지우지 않습니다.\n\"\"\"\nfrom __future__ import annotations\n\nimport io\nimport json\nimport os\nimport random\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageOps\nimport torch\nfrom torch.utils.data import Dataset, DataLoader\nfrom torchvision import transforms as T\n\nfrom src.safe_crop import sample_window\n\n\ndef letterbox(image, size):\n    \"\"\"가로세로비를 지키고 화각을 통째로 남깁니다.\"\"\"\n    image = ImageOps.contain(image, (size, size), Image.Resampling.BILINEAR)\n    result = Image.new('RGB', (size, size), (128, 128, 128))\n    result.paste(image, ((size-image.width)//2, (size-image.height)//2))\n    return result\n\n\nclass OriginalDataset(Dataset):\n    def __init__(self, df, cfg, *, train=False, mode='safe', classes=None,\n                 mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225),\n                 scale=(0.35, 1.0), ratio=(0.85, 1.18), padding=0.05):\n        if mode not in ('safe', 'random', 'full'):\n            raise ValueError(f'Unknown crop mode: {mode}')\n        self.df = df.reset_index(drop=True).copy()\n        self.cfg, self.train, self.mode = cfg, train, mode\n        self.scale, self.ratio, self.padding = scale, ratio, padding\n        self.classes = classes or [f'A{i}' for i in range(1, 8)]\n        mapping = {c: i for i, c in enumerate(self.classes)}\n        if not self.df.label.isin(self.classes).all():\n            raise ValueError('Unknown labels in original dataset')\n        self.targets = self.df.label.map(mapping).to_numpy(dtype=np.int64)\n        # 로짓 지문이 쓰는 경로에 전처리 버전을 넣습니다 — 설정이 다른 실행에 이어붙지 않게.\n        self.paths = [f'original-letterbox-v1:{p}' for p in self.df.image_path]\n        self._archives = {}\n        self._pid = os.getpid()\n        self.normalize = T.Compose([T.ToTensor(), T.Normalize(mean, std)])\n        self.color = T.ColorJitter(brightness=cfg.color_jitter,\n                                   contrast=cfg.color_jitter,\n                                   saturation=cfg.color_jitter*0.5,\n                                   hue=cfg.hue_jitter)\n\n    def __len__(self):\n        return len(self.df)\n\n    def __getstate__(self):\n        state = self.__dict__.copy()\n        state['_archives'] = {}\n        return state\n\n    def close(self):\n        for z in self._archives.values():\n            z.close()\n        self._archives = {}\n\n    def __getitem__(self, index):\n        # ZIP 핸들은 seek 를 쓰므로 워커끼리 **절대** 공유하면 안 됩니다.\n        if self._pid != os.getpid():\n            self.close()\n            self._pid = os.getpid()\n        row = self.df.iloc[index]\n        path = row.zip_path\n        if isinstance(path, str) and path:\n            if path not in self._archives:\n                self._archives[path] = zipfile.ZipFile(path)\n            raw = self._archives[path].read(row.zip_member)\n        else:\n            raw = Path(row.image_path).read_bytes()\n        with Image.open(io.BytesIO(raw)) as im:\n            image = im.convert('RGB')\n        if image.size != (row.img_w, row.img_h):\n            raise ValueError(f'Original dimensions changed: {row.image_path}')\n        image = self.prepare_image(image, row)\n        if self.train:\n            if self.mode == 'safe':\n                boxes = json.loads(row.boxes) if isinstance(row.boxes, str) else row.boxes\n                image = image.crop(sample_window(*image.size, boxes, scale=self.scale,\n                                                ratio=self.ratio, padding=self.padding))\n            elif self.mode == 'random':\n                top, left, h, w = T.RandomResizedCrop.get_params(image, self.scale, self.ratio)\n                image = image.crop((left, top, left+w, top+h))\n            if random.random() < self.cfg.hflip:\n                image = ImageOps.mirror(image)\n            if random.random() < self.cfg.vflip:\n                image = ImageOps.flip(image)\n            image = self.color(image)\n        x = self.normalize(letterbox(image, self.cfg.img_size))\n        return x, int(self.targets[index])\n\n    def prepare_image(self, image, row):\n        return image\n\n\ndef build_original_loaders(train_df, val_df, cfg, model=None, *, mode='safe',\n                           classes=None, scale=(0.35, 1.0), padding=0.05):\n    \"\"\"`src.train.fit` 에 그대로 물립니다. 배치 가림(occlusion)은 명시적으로 거부합니다.\"\"\"\n    if cfg.mixup_alpha or cfg.cutmix_alpha:\n        raise ValueError('Original crop comparison requires mixup/cutmix disabled')\n    if cfg.balance_strategy not in ('none', 'class_weight', 'weighted_sampler'):\n        raise ValueError('Unsupported original-image sampler')\n    pretrained = getattr(model, 'pretrained_cfg', {}) or {}\n    common = dict(classes=classes, mean=pretrained.get('mean', (0.485, 0.456, 0.406)),\n                  std=pretrained.get('std', (0.229, 0.224, 0.225)),\n                  mode=mode, scale=scale, padding=padding)\n    ds_tr = OriginalDataset(train_df, cfg, train=True, **common)\n    ds_va = OriginalDataset(val_df, cfg, train=False, **common)\n    sampler = None\n    if cfg.balance_strategy == 'weighted_sampler':\n        from src.data import weighted_sampler\n        sampler = weighted_sampler(ds_tr)\n    nw = cfg.resolved_num_workers()\n    options = dict(num_workers=nw, pin_memory=torch.cuda.is_available(),\n                   persistent_workers=nw > 0)\n    # RAM 에 쌓이는 배치를 제한합니다 — 워커마다 ZIP 목차까지 물고 있습니다.\n    if nw:\n        options['prefetch_factor'] = 2\n    bs = cfg.resolved_batch_size()\n    dl_tr = DataLoader(ds_tr, batch_size=bs, shuffle=sampler is None,\n                       sampler=sampler, drop_last=False, **options)\n    dl_va = DataLoader(ds_va, batch_size=bs, shuffle=False, **options)\n    return dl_tr, dl_va, ds_tr, ds_va\n", "src/roi_data.py": "\"\"\"고정 ROI vs 병변 근처 random ROI — 검증은 둘 다 고정 ROI (STEP 44 후속).\"\"\"\nimport json\nimport math\nimport random\n\nfrom src.original_data import OriginalDataset\nfrom src.safe_crop import sample_window\n\n\ndef roi_window(width, height, boxes, *, augment=False, rng=None):\n    rng = rng or random\n    # 좌표 검사는 safe_crop 것을 그대로 씁니다. 주석이 없으면 두 방법 모두 원본 전체.\n    sample_window(width, height, boxes, attempts=1, rng=random.Random(0))\n    if not boxes:\n        return (0, 0, width, height)\n    left = max(0, min(b[0] - (b[2]-b[0])*.05 for b in boxes))\n    top = max(0, min(b[1] - (b[3]-b[1])*.05 for b in boxes))\n    right = min(width, max(b[2] + (b[2]-b[0])*.05 for b in boxes))\n    bottom = min(height, max(b[3] + (b[3]-b[1])*.05 for b in boxes))\n    side = max(320, math.ceil(right)-math.floor(left), math.ceil(bottom)-math.floor(top))\n    if augment:\n        side = math.ceil(side * rng.uniform(1.0, 1.25))\n    w, h = min(width, side), min(height, side)\n    xmin, xmax = max(0, math.ceil(right-w)), min(width-w, math.floor(left))\n    ymin, ymax = max(0, math.ceil(bottom-h)), min(height-h, math.floor(top))\n    if augment:\n        x, y = rng.randint(xmin, xmax), rng.randint(ymin, ymax)\n    else:\n        x = min(xmax, max(xmin, round((left+right-w)/2)))\n        y = min(ymax, max(ymin, round((top+bottom-h)/2)))\n    return x, y, x+w, y+h\n\n\nclass ROIDataset(OriginalDataset):\n    def __init__(self, *args, mode='fixed', **kwargs):\n        if mode not in ('fixed', 'safe'):\n            raise ValueError(mode)\n        self.roi_mode = mode\n        super().__init__(*args, mode='full', **kwargs)\n\n    def prepare_image(self, image, row):\n        boxes = json.loads(row.boxes) if isinstance(row.boxes, str) else row.boxes\n        return image.crop(roi_window(*image.size, boxes,\n                                    augment=self.train and self.roi_mode == 'safe'))\n", "src/photo_data.py": "\"\"\"고정 ROI 를 고정해 두고 **`photometric` 만** 켜고 끄는 복원 확인 (이력 감사 ①).\n\nSTEP 44·45 파일럿에는 채택 레시피의 `photometric` 이 **빠져 있었습니다**.\n그래서 *\"safe 가 졌다\"* 가 crop 탓인지 레시피 탓인지 안 갈립니다.\n\n여기서는 **두 팔 다 고정 ROI** 로 두고 학습 증강만 다르게 합니다 —\n바뀌는 것이 하나뿐이라 그 하나의 효과가 나옵니다.\n⚠️ LR·EMA·기간까지 같이 바꾸면 다시 못 가릅니다.\n\n연산자와 확률은 `src.data` 의 `photometric` 프리셋과 **같습니다**.\"\"\"\nimport numpy as np\nfrom PIL import Image\nfrom src.roi_data import ROIDataset\n\n\nclass PhotoNormalize:\n    def __init__(self, normalize, seed):\n        import albumentations as A\n        self.normalize = normalize\n        # 연산자·확률을 src.data 의 photometric 프리셋과 똑같이 맞춥니다.\n        self.aug = A.Compose([\n            A.CLAHE(clip_limit=2.0, p=.2),\n            A.OneOf([A.GaussianBlur(blur_limit=(3,7)),\n                     A.MotionBlur(blur_limit=(3,7))], p=.3),\n            A.GaussNoise(p=.25),\n            A.ImageCompression(quality_range=(40,90), p=.3),\n        ], seed=seed)\n        self.worker = None\n\n    def __call__(self, image):\n        from torch.utils.data import get_worker_info\n        info = get_worker_info()\n        if info is not None and self.worker != info.id:\n            self.aug.set_random_seed(info.seed % (2**32))\n            self.worker = info.id\n        return self.normalize(Image.fromarray(self.aug(image=np.asarray(image))['image']))\n\n\nclass PhotoDataset(ROIDataset):\n    def __init__(self, *args, mode='fixed', **kwargs):\n        if mode not in ('fixed','photo'):\n            raise ValueError(mode)\n        super().__init__(*args, mode='fixed', **kwargs)\n        if self.train and mode == 'photo':\n            self.normalize = PhotoNormalize(self.normalize, self.cfg.seed + self.cfg.photo_epoch)\n", "tools/kaggle_safe_crop.py": "\"\"\"시간 예산이 있는 1단계 짝 비교 — epoch 경계에서 이어받습니다 (STEP 44).\n\n    python tools/kaggle_safe_crop.py --data DATASET_ROOT --out OUTPUT_DIR\n\n`--profile stage2` (STEP 49) 는 같은 패키지에서 A7 을 빼고 병변 6종을 배웁니다 —\n두 팔 `fixed`(고정 ROI) / `safe`(병변 보존 random ROI), 매 epoch clean + 위치 교란 평가.\n\n이 실행기와 `src/` 스냅샷은 파일럿 데이터에 **같이 담겨** 나갑니다 —\n캐글에서 git clone 도 API 키도 필요 없게.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport random\nimport sys\nimport time\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.metrics import balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score\nfrom torch.utils.data import DataLoader\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\n\nfrom src.config import CFG, CLASSES, MORPH_GROUP_KEEP_A6\nfrom src.original_data import OriginalDataset\nfrom src.roi_data import ROIDataset, roi_window\n\nSTAGE2_SHIFT = 0.20  # STEP 47 과 같은 정의: 고정 ROI 를 변 길이의 20% 만큼 오른쪽·아래로\n\n\nclass ShiftedROIDataset(ROIDataset):\n    \"\"\"위치 교란 평가용. 병변을 보존하도록 제한하지 않습니다 — 창이 경계에서만 멈춥니다.\"\"\"\n\n    def prepare_image(self, image, row):\n        boxes = json.loads(row.boxes) if isinstance(row.boxes, str) else row.boxes\n        x0, y0, x1, y1 = roi_window(*image.size, boxes)\n        dx = min(round((x1-x0)*STAGE2_SHIFT), image.width-x1)\n        dy = min(round((y1-y0)*STAGE2_SHIFT), image.height-y1)\n        return image.crop((x0+dx, y0+dy, x1+dx, y1+dy))\n\n\ndef seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef atomic_save(value, path):\n    temporary = path.with_suffix(path.suffix + '.part')\n    torch.save(value, temporary)\n    temporary.replace(path)\n\n\ndef load_checkpoint(path, device='cpu'):\n    return torch.load(path, map_location=device, weights_only=True)\n\n\ndef load_data(root, profile='original'):\n    metadata = json.loads((root / 'pilot_package.json').read_text())\n    raw = (root / 'pilot_manifest.parquet').read_bytes()\n    if hashlib.sha256(raw).hexdigest() != metadata['pilot_manifest_sha256']:\n        raise ValueError('Pilot manifest differs from the packaged version')\n    for name, digest in metadata['code_sha256'].items():\n        if hashlib.sha256((root / name).read_bytes()).hexdigest() != digest:\n            raise ValueError(f'Packaged source changed: {name}')\n    df = pd.read_parquet(root / 'pilot_manifest.parquet')\n    if not df.pilot_split.isin(['train', 'val']).all():\n        raise ValueError('Unexpected split: holdout must not be included')\n    for member in df.image_path:\n        relative = Path(member)\n        if relative.is_absolute() or '..' in relative.parts:\n            raise ValueError('Nonportable image path')\n    df['image_path'] = df.image_path.map(lambda p: str(root / p))\n    df['original_label'] = df.label\n    if profile != 'stage2':\n        df['label'] = df.label.where(df.label == 'A7', 'ABNORMAL')\n    tr, va = (df[df.pilot_split == name].reset_index(drop=True) for name in ['train', 'val'])\n    for col in ['group', 'animal_id', 'sha256']:\n        if set(tr[col]) & set(va[col]):\n            raise ValueError(f'Train/validation overlap in {col}')\n    if len(tr) != metadata['train_rows'] or len(va) != metadata['val_rows']:\n        raise ValueError('Pilot row counts changed')\n    if profile == 'stage2':\n        # 1단계를 통과했다고 치고 병변만 넘어온 상황. 패키지 행 수 검사 **뒤에** 거릅니다.\n        tr, va = (d[d.label.isin(CLASSES)].reset_index(drop=True) for d in [tr, va])\n        expected = {s: sum(metadata['counts'][c][s] for c in CLASSES) for s in ['train', 'val']}\n        if len(tr) != expected['train'] or len(va) != expected['val']:\n            raise ValueError('Stage-2 row counts differ from the package counts')\n    return tr, va, metadata\n\n\ndef make_model(smoke, pretrained=False, profile=\"original\"):\n    n_classes = len(CLASSES) if profile == \"stage2\" else 2\n    if profile in (\"roi\", \"photo\", \"stage2\") and not smoke:\n        import timm\n        return timm.create_model(\"tf_efficientnetv2_s.in21k_ft_in1k\", pretrained=pretrained,\n                                 num_classes=n_classes, drop_rate=0.2, drop_path_rate=0.1)\n    if smoke:\n        return torch.nn.Sequential(torch.nn.Conv2d(3, 8, 3, stride=2),\n                                   torch.nn.ReLU(), torch.nn.AdaptiveAvgPool2d(1),\n                                   torch.nn.Flatten(), torch.nn.Linear(8, n_classes))\n    from torchvision.models import resnet50, ResNet50_Weights\n    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)\n    model.fc = torch.nn.Linear(model.fc.in_features, 2)\n    return model\n\n\ndef metrics(probability, target, original_labels):\n    prediction = (probability >= 0.5).astype(int)\n    tn, fp, fn, tp = confusion_matrix(target, prediction, labels=[0, 1]).ravel()\n    result = {'macro_f1': float(f1_score(target, prediction, labels=[0, 1], average='macro', zero_division=0)),\n              'balanced_accuracy': float(balanced_accuracy_score(target, prediction)),\n              'abnormal_recall': float(tp/max(tp+fn, 1)),\n              'normal_specificity': float(tn/max(tn+fp, 1)),\n              'auroc': float(roc_auc_score(target, probability)) if len(set(target)) == 2 else None}\n    for label in sorted(set(original_labels)):\n        mask = original_labels == label\n        result[f'{label}_correct_rate'] = float((prediction[mask] == target[mask]).mean())\n    return result\n\n\ndef metrics_stage2(probability, target):\n    \"\"\"6종 + 계열 4군(`MORPH_GROUP_KEEP_A6`, 확률 합 → argmax). 문턱 없이 argmax 만.\"\"\"\n    prediction = probability.argmax(1)\n    labels = list(range(len(CLASSES)))\n    result = {'macro_f1': float(f1_score(target, prediction, labels=labels, average='macro', zero_division=0)),\n              'accuracy': float((prediction == target).mean())}\n    for i, code in enumerate(CLASSES):\n        mask = target == i\n        result[f'{code}_recall'] = float((prediction[mask] == i).mean()) if mask.any() else None\n    groups = list(dict.fromkeys(MORPH_GROUP_KEEP_A6[c] for c in CLASSES))\n    member = np.array([groups.index(MORPH_GROUP_KEEP_A6[c]) for c in CLASSES])\n    group_probability = np.stack([probability[:, member == g].sum(1) for g in range(len(groups))], 1)\n    group_target, group_prediction = member[target], group_probability.argmax(1)\n    result['group4_macro_f1'] = float(f1_score(group_target, group_prediction, labels=list(range(len(groups))),\n                                               average='macro', zero_division=0))\n    result['group4_accuracy'] = float((group_prediction == group_target).mean())\n    a4, a1 = CLASSES.index('A4'), CLASSES.index('A1')\n    result['A4_to_A1'] = float((prediction[target == a4] == a1).mean()) if (target == a4).any() else None\n    return result\n\n\ndef one_epoch(model, optimizer, scaler, tr, va, cfg, mode, epoch, device,\n              deadline, max_train_batches=None):\n    \"\"\"마감을 넘기면 None — 부르는 쪽이 직전에 커밋된 epoch 를 그대로 유지합니다.\"\"\"\n    seed_everything(cfg.seed + epoch)\n    model.train()\n    dataset = OriginalDataset\n    if getattr(cfg, 'experiment_profile', 'original') == 'roi':\n        dataset = ROIDataset\n    if getattr(cfg, 'experiment_profile', 'original') == 'photo':\n        from src.photo_data import PhotoDataset\n        dataset = PhotoDataset\n        cfg.photo_epoch = epoch\n    stage2 = getattr(cfg, 'experiment_profile', 'original') == 'stage2'\n    if stage2:\n        dataset = ROIDataset\n    classes = list(CLASSES) if stage2 else ['A7', 'ABNORMAL']\n    ds_tr = dataset(tr, cfg, train=True, mode=mode, classes=classes)\n    ds_va = dataset(va, cfg, train=False, mode=mode, classes=classes)\n    # 위치 교란은 두 팔 다 **고정 ROI 에서 출발**합니다 — 학습 팔과 무관하게 같은 창.\n    ds_shift = ShiftedROIDataset(va, cfg, train=False, mode='fixed', classes=classes) if stage2 else None\n    generator = torch.Generator().manual_seed(cfg.seed + epoch)\n    options = dict(batch_size=cfg.batch_size, num_workers=cfg.num_workers,\n                   pin_memory=device == 'cuda')\n    if cfg.num_workers:\n        options['prefetch_factor'] = 2\n    dl_tr = DataLoader(ds_tr, shuffle=True, generator=generator, **options)\n    dl_va = DataLoader(ds_va, shuffle=False, **options)\n    counts = np.bincount(ds_tr.targets, minlength=len(classes))\n    weights = torch.tensor(len(ds_tr)/(len(classes)*np.maximum(counts, 1)), dtype=torch.float32, device=device)\n    criterion = torch.nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)\n    started = time.monotonic()\n    losses, trained = 0.0, 0\n    try:\n        for batch, (x, y) in enumerate(dl_tr):\n            if time.monotonic() >= deadline:\n                return None\n            x, y = x.to(device), y.to(device)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device, enabled=device == 'cuda'):\n                loss = criterion(model(x), y)\n            if not torch.isfinite(loss):\n                raise ValueError('Nonfinite training loss')\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            scaler.step(optimizer)\n            scaler.update()\n            losses += float(loss.detach()) * len(y)\n            trained += len(y)\n            if (batch+1) % 100 == 0:\n                print(f'{mode} epoch {epoch+1}: {trained}/{len(ds_tr)} photos', flush=True)\n            if max_train_batches and batch+1 >= max_train_batches:\n                break\n        model.eval()\n        probabilities, targets = [], []\n        blurred = {1: [], 2: []} if getattr(cfg, 'experiment_profile', '') == 'photo' else {}\n        with torch.no_grad():\n            for x, y in dl_va:\n                if time.monotonic() >= deadline:\n                    return None\n                with torch.autocast(device_type=device, enabled=device == 'cuda'):\n                    logits = model(x.to(device))\n                softmax = logits.float().softmax(1).cpu()\n                probabilities.extend(softmax.tolist() if stage2 else softmax[:, 1].tolist())\n                targets.extend(y.tolist())\n                if blurred:\n                    from torchvision.transforms.functional import gaussian_blur\n                    for sigma in blurred:\n                        if time.monotonic() >= deadline:\n                            return None\n                        with torch.autocast(device_type=device, enabled=device == 'cuda'):\n                            altered = model(gaussian_blur(x.to(device), [13,13], [float(sigma)]*2))\n                        blurred[sigma].extend(altered.float().softmax(1)[:,1].cpu().tolist())\n            shifted, shifted_targets = [], []\n            if ds_shift is not None:\n                for x, y in DataLoader(ds_shift, shuffle=False, **options):\n                    if time.monotonic() >= deadline:\n                        return None\n                    with torch.autocast(device_type=device, enabled=device == 'cuda'):\n                        logits = model(x.to(device))\n                    shifted.extend(logits.float().softmax(1).cpu().tolist())\n                    shifted_targets.extend(y.tolist())\n        if device == 'cuda':\n            torch.cuda.synchronize()\n        if stage2:\n            if shifted_targets != targets:\n                raise ValueError('Shifted validation rows are not aligned with clean rows')\n            result = metrics_stage2(np.asarray(probabilities), np.asarray(targets))\n            shift_scores = metrics_stage2(np.asarray(shifted), np.asarray(targets))\n            result.update({f'shift_{k}': v for k, v in shift_scores.items()})\n            result['shift_drop_rel'] = (result['macro_f1']-result['shift_macro_f1'])/max(result['macro_f1'], 1e-9)\n        else:\n            result = metrics(np.asarray(probabilities), np.asarray(targets), va.original_label.to_numpy())\n        for sigma, values in blurred.items():\n            scores = metrics(np.asarray(values), np.asarray(targets), va.original_label.to_numpy())\n            result.update({f'blur{sigma}_{k}': v for k,v in scores.items()})\n        result.update(epoch=epoch+1, mode=mode, train_loss=losses/max(trained, 1),\n                      elapsed_sec=time.monotonic()-started, train_rows=trained, val_rows=len(va))\n        return result, np.asarray(probabilities)\n    finally:\n        ds_tr.close()\n        ds_va.close()\n        if ds_shift is not None:\n            ds_shift.close()\n\n\ndef write_comparison(output, histories):\n    records = [r for history in histories.values() for r in history]\n    table = pd.DataFrame(records) if records else pd.DataFrame(columns=['mode', 'epoch', 'macro_f1'])\n    table.to_csv(output / 'history.csv', index=False)\n    baseline_mode, treatment_mode = list(histories)\n    common = min(len(histories[baseline_mode]), len(histories[treatment_mode]))\n    summary = {'completed_epochs': {k: len(v) for k, v in histories.items()},\n               'common_epoch': common, 'comparison': None,\n               'note': 'Exploratory pilot; compare only the same epoch. No holdout evaluation.'}\n    if common:\n        baseline, safe = (histories[m][common-1] for m in [baseline_mode, treatment_mode])\n        summary['comparison'] = {key: {baseline_mode: baseline[key], treatment_mode: safe[key],\n                                       f'{treatment_mode}_minus_{baseline_mode}': safe[key]-baseline[key]}\n                                 for key in [k for k in ['macro_f1', 'abnormal_recall', 'normal_specificity', 'balanced_accuracy', 'auroc', 'blur1_macro_f1', 'blur2_macro_f1', 'blur1_auroc', 'blur2_auroc',\n                                                         'accuracy', 'group4_macro_f1', 'group4_accuracy', 'A6_recall', 'A4_recall', 'A5_recall', 'A4_to_A1',\n                                                         'shift_macro_f1', 'shift_group4_accuracy', 'shift_A6_recall', 'shift_drop_rel'] if k in baseline and baseline[k] is not None and safe[k] is not None]}\n    (output / 'comparison.json').write_text(json.dumps(summary, indent=2))\n    print(json.dumps(summary, indent=2), flush=True)\n    return summary\n\n\ndef export_resume(output):\n    # 사진과 데이터셋은 캐글 출력에 안 넣습니다 — 작은 실행 산출물만.\n    profile = json.loads((output / 'protocol.json').read_text()).get('profile', 'original')\n    archive = output.parent / {'roi':'roi_crop_pilot_resume.zip', 'photo':'photo_crop_pilot_resume.zip',\n                               'stage2':'stage2_crop_pilot_resume.zip'}.get(profile,'safe_crop_pilot_resume.zip')\n    temporary = archive.with_suffix('.zip.part')\n    with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_STORED) as z:\n        for path in sorted(output.rglob('*')):\n            if path.is_file() and not path.name.endswith('.part'):\n                z.write(path, path.relative_to(output))\n    temporary.replace(archive)\n    return archive\n\n\ndef run(args):\n    started = time.monotonic()\n    profile = getattr(args, 'profile', 'original')\n    modes = {'roi':['fixed','safe'], 'photo':['fixed','photo'], 'stage2':['fixed','safe']}.get(profile,['random','safe'])\n    device = 'cuda' if torch.cuda.is_available() else 'cpu'\n    if device != 'cuda' and not args.smoke:\n        raise RuntimeError('Select a GPU accelerator; use --smoke only for local tests')\n    if not (0 < args.hours <= 7.5) or not (1 <= args.epochs <= 5):\n        raise ValueError('Pilot budget: 0 < hours <= 7.5 and 1 <= epochs <= 5')\n    args.out.mkdir(parents=True, exist_ok=True)\n    tr, va, metadata = load_data(args.data, profile)\n    if args.smoke:\n        tr = tr.groupby('original_label', group_keys=False).head(2).reset_index(drop=True)\n        va = va.groupby('original_label', group_keys=False).head(1).reset_index(drop=True)\n    cfg = CFG(seed=42, img_size=32 if args.smoke else (384 if profile in ('roi','photo','stage2') else 288), batch_size=args.batch_size,\n              num_workers=args.workers, epochs=args.epochs, lr=3e-4,\n              rotate_deg=0, random_erasing=0, amp=device == 'cuda')\n    protocol = {'version': 1, 'manifest_sha256': metadata['pilot_manifest_sha256'],\n                'code_sha256': metadata['code_sha256'], 'smoke_only': args.smoke,\n                'model': 'tiny' if args.smoke else 'torchvision_resnet50_IMAGENET1K_V2',\n                'cfg': cfg.to_dict(), 'crop_scale': [0.35, 1.0], 'padding': 0.05,\n                'validation': 'full-frame letterbox', 'task': 'stage1',\n                'schedule': 'paired alternating epochs; fixed cosine horizon',\n                'torch_version': str(torch.__version__)}\n    if profile in ('roi','photo','stage2'):\n        import timm\n        from PIL import ImageFile\n        cfg.model_name = 'tf_efficientnetv2_s.in21k_ft_in1k'\n        cfg.experiment_profile = profile\n        protocol.update(profile=profile, version=2,\n                        cfg=cfg.to_dict(),\n                        load_truncated_images=ImageFile.LOAD_TRUNCATED_IMAGES,\n                        model='tiny' if args.smoke else 'tf_efficientnetv2_s.in21k_ft_in1k',\n                        timm_version=timm.__version__, validation='fixed lesion ROI letterbox',\n                        crop_scale=None, roi_min_side=320, roi_side_multiplier=[1.0, 1.25],\n                        modes=modes, augmentation='shared flip/color only; no rotation/erasing',\n                        missing_boxes='full frame in both arms',\n                        runtime_code_sha256={name: hashlib.sha256(\n                            (Path(__file__).resolve().parents[1] / name).read_bytes()).hexdigest()\n                            for name in ['tools/kaggle_safe_crop.py', 'src/roi_data.py',\n                                         'src/original_data.py', 'src/safe_crop.py', 'src/config.py']})\n    if profile == 'photo':\n        import albumentations as A\n        protocol.update(roi_side_multiplier=[1.0,1.0],\n                        augmentation='fixed ROI both arms; photo adds historical photometric operators after letterbox',\n                        photometric={'clahe_p':.2,'blur_p':.3,'noise_p':.25,'jpeg_p':.3},\n                        albumentations_version=A.__version__,\n                        robustness='all val every epoch: clean + tensor Gaussian kernel13 sigma1/sigma2 reflect padding')\n        protocol['runtime_code_sha256']['src/photo_data.py'] = hashlib.sha256(\n            (Path(__file__).resolve().parents[1]/'src/photo_data.py').read_bytes()).hexdigest()\n    if profile == 'stage2':\n        protocol.update(version=3, task='stage2', classes=list(CLASSES),\n                        rows={'train': len(tr), 'val': len(va)},\n                        excluded='A7 rows dropped after the package row-count check',\n                        model='tiny' if args.smoke else 'tf_efficientnetv2_s.in21k_ft_in1k num_classes=6',\n                        validation='fixed lesion ROI letterbox (clean) + same ROI moved 20% of its side right/down, clipped at the image edge (shift)',\n                        shift_fraction=STAGE2_SHIFT, groups=dict(MORPH_GROUP_KEEP_A6),\n                        decision='argmax only; no stage-1 probability, so coverage is not measured here')\n    protocol_path = args.out / 'protocol.json'\n    if protocol_path.exists():\n        if json.loads(protocol_path.read_text()) != json.loads(json.dumps(protocol)):\n            raise ValueError('Resume protocol differs; keep the same data, settings and source snapshot')\n    else:\n        if any(args.out.iterdir()):\n            raise ValueError('Nonempty output directory has no protocol')\n        protocol_path.write_text(json.dumps(protocol, indent=2))\n    initial_path = args.out / 'initial.pt'\n    if not initial_path.exists():\n        if any(args.out.glob('*_last.pt')):\n            raise ValueError('Missing common initialization in resumed run')\n        seed_everything(cfg.seed)\n        initial = make_model(args.smoke, pretrained=not args.smoke, profile=profile)\n        atomic_save(initial.state_dict(), initial_path)\n        del initial\n    initial_digest = hashlib.sha256(initial_path.read_bytes()).hexdigest()\n    histories = {}\n    for mode in modes:\n        last = args.out / f'{mode}_last.pt'\n        state = load_checkpoint(last) if last.exists() else None\n        if state and state['initial_sha256'] != initial_digest:\n            raise ValueError('Initialization mismatch')\n        histories[mode] = state['history'] if state else []\n    # 체크포인트·내보내기에 10분을 남깁니다. --hours 는 이번 호출의 벽시계 시간입니다.\n    reserve = min(600, args.hours*3600*0.1)\n    deadline = started + args.hours*3600 - reserve\n    reason = 'completed'\n    try:\n        for epoch in range(args.epochs):\n            order = modes if epoch % 2 == 0 else list(reversed(modes))\n            for mode in order:\n                if len(histories[mode]) > epoch:\n                    continue\n                if len(histories[mode]) != epoch:\n                    raise ValueError('Noncontiguous checkpoint history')\n                observed = [r['elapsed_sec'] for h in histories.values() for r in h]\n                estimate = max(observed[-4:]) * 1.35 if observed else 0\n                pending_in_pair = sum(len(history) <= epoch for history in histories.values())\n                if time.monotonic() + estimate * pending_in_pair >= deadline:\n                    reason = 'time_budget_before_epoch'\n                    return write_comparison(args.out, histories)\n                model = make_model(args.smoke, profile=profile).to(device)\n                optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)\n                scaler = torch.amp.GradScaler('cuda', enabled=device == 'cuda')\n                last = args.out / f'{mode}_last.pt'\n                if last.exists():\n                    state = load_checkpoint(last, device)\n                    model.load_state_dict(state['model'])\n                    optimizer.load_state_dict(state['optimizer'])\n                    scaler.load_state_dict(state['scaler'])\n                    del state\n                else:\n                    model.load_state_dict(load_checkpoint(initial_path))\n                # epoch 로 색인한 스케줄이라 중단된 epoch 를 다시 돌려도 결정론적입니다.\n                lr = cfg.lr * (0.1 + 0.9*(1+math.cos(math.pi*epoch/cfg.epochs))/2)\n                for group in optimizer.param_groups:\n                    group['lr'] = lr\n                result = one_epoch(model, optimizer, scaler, tr, va, cfg, mode, epoch,\n                                   device, deadline)\n                if result is None:\n                    reason = 'time_budget_during_epoch_replay_required'\n                    return write_comparison(args.out, histories)\n                record, probabilities = result\n                histories[mode].append(record)\n                atomic_save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),\n                             'scaler': scaler.state_dict(), 'history': histories[mode],\n                             'initial_sha256': initial_digest}, last)\n                if record['macro_f1'] >= max(r['macro_f1'] for r in histories[mode]):\n                    atomic_save({'model': model.state_dict(), 'epoch': epoch+1,\n                                 'initial_sha256': initial_digest}, args.out / f'{mode}_best.pt')\n                # epoch 마다 작은 예측을 남겨야 이어받은 뒤에도 **공통 epoch** 비교가 됩니다.\n                np.savez_compressed(args.out / f'{mode}_epoch{epoch+1}_val.npz',\n                                    probability=probabilities, sha256=va.sha256.to_numpy(dtype=str),\n                                    original_label=va.original_label.to_numpy(dtype=str))\n                print(json.dumps(record, indent=2), flush=True)\n                print(f'Measured epoch {record[\"elapsed_sec\"]/60:.1f} min; '\n                      f'10 epochs at this rate ≈ {record[\"elapsed_sec\"]*10/3600:.2f} h (estimate)', flush=True)\n                write_comparison(args.out, histories)\n                del model, optimizer, scaler\n                if device == 'cuda':\n                    torch.cuda.empty_cache()\n        return write_comparison(args.out, histories)\n    except BaseException:\n        reason = 'error_or_interruption'\n        raise\n    finally:\n        (args.out / 'session.json').write_text(json.dumps({\n            'stop_reason': reason, 'elapsed_sec': time.monotonic()-started,\n            'budget_hours': args.hours, 'device': device, 'smoke_only': args.smoke,\n            'initial_sha256': initial_digest}, indent=2))\n        print('Resume bundle:', export_resume(args.out), flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument('--data', type=Path, required=True)\n    parser.add_argument('--out', type=Path, required=True)\n    parser.add_argument('--hours', type=float, default=7.0)\n    parser.add_argument('--epochs', type=int, default=5)\n    parser.add_argument('--batch-size', type=int, default=32)\n    parser.add_argument('--workers', type=int, default=2)\n    parser.add_argument('--profile', choices=['original', 'roi', 'photo', 'stage2'], default='original')\n    parser.add_argument('--smoke', action='store_true')\n    run(parser.parse_args())\n\n\nif __name__ == '__main__':\n    main()\n"}

for name, source in FILES.items():
    path = CODE / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source)
# 이전 실행과 동일한 Pillow 복구 설정: subprocess 와 worker 모두 적용.
(CODE / 'sitecustomize.py').write_text('from PIL import ImageFile\nImageFile.LOAD_TRUNCATED_IMAGES = True\n')
ENV = dict(os.environ)
ENV['PYTHONPATH'] = str(CODE)
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'
ENV['NO_ALBUMENTATIONS_UPDATE'] = '1'
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'timm==1.0.29'], check=True)
# 검증한 버전으로 고정. 이미 설치된 구버전도 교체하고 학습은 새 subprocess에서 시작.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'albumentations==2.0.8'], check=True)
# 실제 사용할 모델의 GPU forward/backward 를 먼저 확인.
probe = """import torch
from tools.kaggle_safe_crop import make_model
assert torch.cuda.is_available(), 'GPU T4를 선택하세요'
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0), flush=True)
m = make_model(False, profile='stage2').cuda().train()
with torch.autocast('cuda'):
    loss = m(torch.randn(2, 3, 64, 64, device='cuda')).float().square().mean()
loss.backward()
torch.cuda.synchronize()
print('GPU probe passed', flush=True)
"""
subprocess.run([sys.executable, '-c', probe], env=ENV, cwd=CODE, check=True)


In [ ]:
candidates = list(Path('/kaggle/input').rglob('pilot_manifest.parquet'))
assert len(candidates) == 1, f'기존 safe-crop-pilot Dataset 하나를 연결하세요: {candidates}'
DATA = candidates[0].parent
# 2단계 실험 결과만 재개. 1단계(roi/photo) ZIP 은 쓰지 않습니다.
# ⚠️ 캐글은 Dataset 을 만들 때 ZIP 을 **자동으로 풀어** 올립니다 — 그러면 ZIP 이 안 보여
# 조용히 처음부터 다시 돌았습니다 (2026-09-11, 34분 손실). 풀린 폴더도 같이 찾습니다.
import shutil
resumes = list(Path('/kaggle/input').rglob('stage2_crop_pilot_resume*.zip'))
extracted = [p.parent for p in Path('/kaggle/input').rglob('protocol.json')
             if json.loads(p.read_text()).get('profile') == 'stage2']
if not OUT.exists():
    if resumes:
        assert len(resumes) == 1, '재개 ZIP 하나만 연결하세요'
        with zipfile.ZipFile(resumes[0]) as z:
            for name in z.namelist():
                assert not Path(name).is_absolute() and '..' not in Path(name).parts
            assert json.loads(z.read('protocol.json')).get('profile') == 'stage2'
            z.extractall(OUT)
    elif extracted:
        assert len(extracted) == 1, f'재개 폴더 하나만 연결하세요: {extracted}'
        shutil.copytree(extracted[0], OUT)
if OUT.exists():
    done = sorted(p.name for p in OUT.glob('*_epoch*_val.npz'))
    print('재개:', OUT, '— 끝난 epoch 파일', done)
    assert done, '재개 폴더에 epoch 결과가 없습니다 — 잘못된 입력'
else:
    print('⚠️ 재개 없음 — 처음부터 돕니다. 이어서 돌리려던 거면 지금 멈추고 입력을 확인하세요')
print('Dataset:', DATA, 'Output:', OUT)


In [ ]:
command = [sys.executable, '-u', str(CODE / 'tools/kaggle_safe_crop.py'),
           '--profile', 'stage2', '--data', str(DATA), '--out', str(OUT),
           '--hours', str(HOURS), '--epochs', str(EPOCHS),
           '--batch-size', str(BATCH_SIZE), '--workers', str(WORKERS)]
subprocess.run(command, check=True, cwd=CODE, env=ENV)


In [ ]:
import pandas as pd
print((OUT / 'comparison.json').read_text())
history = pd.read_csv(OUT / 'history.csv')
# clean 만 보면 crop 의 값어치를 못 봅니다 — 위치 교란(shift) 열을 같이 봅니다.
columns = [c for c in ['mode', 'epoch', 'macro_f1', 'group4_accuracy', 'A6_recall', 'A4_recall',
                       'shift_macro_f1', 'shift_drop_rel', 'shift_group4_accuracy',
                       'shift_A6_recall', 'elapsed_sec'] if c in history.columns]
display(history[columns])
print('다운로드: /kaggle/working/stage2_crop_pilot_resume.zip')
